<a href="https://colab.research.google.com/github/ramu-49/Multi-Class-Legal-Case-Outcome-Classification-using-TF-IDF-and-Tuned-Logistic-Regression/blob/main/Legal_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Attempt to load the data by skipping problematic lines
try:
    df = pd.read_csv('legal_text_classification.csv',
                    engine='python',
                    on_bad_lines='skip')
    print("Dataset loaded successfully by skipping malformed lines.")
except Exception as e:
    print(f"Failed to load: {e}")

# If the error persists, use a more flexible quoting strategy
if 'df' not in locals():
    df = pd.read_csv('legal_text_classification.csv',
                     quoting=1, # csv.QUOTE_ALL
                     error_bad_lines=False,
                     warn_bad_lines=True)

Dataset loaded successfully by skipping malformed lines.


In [3]:
# Splitting the dataset into 80% training and 20% testing
# random_state ensures reproducibility
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)

# Verify the split
print(f"Total rows successfully loaded: {len(df)}")
print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(test_df)}")

# Display the first few rows of the training set to ensure it's correct
print(train_df[['case_id', 'case_outcome']].head())

Total rows successfully loaded: 24984
Training set size: 19987
Testing set size: 4997
         case_id case_outcome
18065  Case18223     followed
24325  Case24546        cited
13151  Case13271        cited
2712    Case2735  referred to
20007  Case20191        cited


In [14]:

train_df.to_csv('legal_train.csv', index=False)
test_df.to_csv('legal_test.csv', index=False)


from google.colab import files
files.download('legal_train.csv')
files.download('legal_test.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import pandas as pd

try:
    df_sample = pd.read_csv('legal_text_classification.csv',
                            engine='python',
                            on_bad_lines='skip',
                            nrows=20)


    df_sample.to_csv('sample_data.csv', index=False)

    print("Successfully created sample_data.csv with 20 rows.")


    from google.colab import files
    files.download('sample_data.csv')

except Exception as e:
    print(f"Error creating sample: {e}")

Successfully created sample_data.csv with 20 rows.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# Check columns
print(df.columns)

# Check for null values
print(df.isnull().sum())

# Check class distribution
print(df['case_outcome'].value_counts())


Index(['case_id', 'case_outcome', 'case_title', 'case_text'], dtype='object')
case_id           0
case_outcome      0
case_title        0
case_text       176
dtype: int64
case_outcome
cited            12219
referred to       4384
applied           2448
followed          2256
considered        1712
discussed         1023
distinguished      608
related            113
affirmed           113
approved           108
Name: count, dtype: int64


In [5]:
# Drop rows where case_text is missing
df = df.dropna(subset=['case_text'])

# Verify removal
print("Total rows after removing null case_text:", len(df))
print(df.isnull().sum())


Total rows after removing null case_text: 24808
case_id         0
case_outcome    0
case_title      0
case_text       0
dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['case_outcome']   # This is the key improvement
)

print("Training set size:", len(train_df))
print("Testing set size:", len(test_df))

# Check class distribution
print("\nTraining Distribution:")
print(train_df['case_outcome'].value_counts(normalize=True))

print("\nTesting Distribution:")
print(test_df['case_outcome'].value_counts(normalize=True))


Training set size: 19846
Testing set size: 4962

Training Distribution:
case_outcome
cited            0.488159
referred to      0.175854
applied          0.098257
followed         0.090799
considered       0.068477
discussed        0.041016
distinguished    0.024287
related          0.004535
approved         0.004333
affirmed         0.004283
Name: proportion, dtype: float64

Testing Distribution:
case_outcome
cited            0.488110
referred to      0.175937
applied          0.098347
followed         0.090689
considered       0.068521
discussed        0.040911
distinguished    0.024385
related          0.004434
approved         0.004434
affirmed         0.004232
Name: proportion, dtype: float64


In [7]:
import string
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stopwords
    text = ' '.join(word for word in text.split() if word not in stop_words)

    return text


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [8]:
train_df['clean_text'] = train_df['case_text'].apply(clean_text)
test_df['clean_text'] = test_df['case_text'].apply(clean_text)

print("Sample cleaned text:\n")
print(train_df[['case_text', 'clean_text']].head(2))


Sample cleaned text:

                                              case_text  \
8076  response to the breach claim for present purpo...   
2027  Comcare v Etheridge and Others [2006] FCAFC 27...   

                                             clean_text  
8076  response breach claim present purposes twofold...  
2027  comcare v etheridge others fcafc fcr branson j...  


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,        # Increased vocabulary size
    ngram_range=(1, 2),       # Unigrams + Bigrams
    min_df=5,                 # Ignore rare words
    max_df=0.9                # Ignore extremely common words
)

X_train = tfidf.fit_transform(train_df['clean_text'])
X_test = tfidf.transform(test_df['clean_text'])

y_train = train_df['case_outcome']
y_test = test_df['case_outcome']

print("TF-IDF matrix shape (Training):", X_train.shape)
print("TF-IDF matrix shape (Testing):", X_test.shape)


TF-IDF matrix shape (Training): (19846, 5000)
TF-IDF matrix shape (Testing): (4962, 5000)


In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=2000,              # Ensure convergence
    multi_class='multinomial',  # True multi-class optimization
    solver='lbfgs',             # Works well for multinomial
    class_weight='balanced'     # Handle class imbalance
)

model.fit(X_train, y_train)

print("Model training completed.")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Model training completed.


In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Overall Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Overall Accuracy: 0.3851269649334946

Confusion Matrix:

[[ 18   0   0   1   0   0   0   0   0   2]
 [  2 165  12  55  54  52  25  63  56   4]
 [  0   1   8   1   3   3   1   2   2   1]
 [ 31 292  32 918 205 175 163 234 328  44]
 [  4  50   6  29 116  36  22  41  33   3]
 [  0  17   5  16  22  91  19  11  20   2]
 [  1   8   2  10  13  16  59   7   3   2]
 [  3  56   7  65  47  34  29 158  47   4]
 [  7  85   3 110  84  78  55  69 366  16]
 [  1   4   0   1   1   1   2   0   0  12]]

Classification Report:

               precision    recall  f1-score   support

     affirmed       0.27      0.86      0.41        21
      applied       0.24      0.34      0.28       488
     approved       0.11      0.36      0.16        22
        cited       0.76      0.38      0.51      2422
   considered       0.21      0.34      0.26       340
    discussed       0.19      0.45      0.26       203
distinguished       0.16      0.49      0.24       121
     followed       0.27      0.35      0.31  

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Define parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 5, 10]
}

base_model = LogisticRegression(
    max_iter=2000,
    solver='lbfgs',
    class_weight='balanced'
)

grid_search = GridSearchCV(
    base_model,
    param_grid,
    cv=3,                # 3-fold cross validation
    scoring='f1_weighted',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)


Best Parameters: {'C': 10}
Best Cross-Validation Score: 0.46102821855287907


In [13]:
final_model = LogisticRegression(
    max_iter=2000,
    solver='lbfgs',
    class_weight='balanced',
    C=10
)

final_model.fit(X_train, y_train)

y_pred_final = final_model.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

print("New Accuracy:", accuracy_score(y_test, y_pred_final))
print("\nNew Classification Report:\n")
print(classification_report(y_test, y_pred_final))


New Accuracy: 0.4494155582426441

New Classification Report:

               precision    recall  f1-score   support

     affirmed       0.34      0.62      0.44        21
      applied       0.27      0.38      0.32       488
     approved       0.17      0.32      0.22        22
        cited       0.75      0.48      0.58      2422
   considered       0.26      0.40      0.32       340
    discussed       0.22      0.38      0.28       203
distinguished       0.23      0.40      0.29       121
     followed       0.28      0.38      0.33       450
  referred to       0.46      0.48      0.47       873
      related       0.27      0.45      0.34        22

     accuracy                           0.45      4962
    macro avg       0.32      0.43      0.36      4962
 weighted avg       0.53      0.45      0.47      4962

